# Line Fitting — Watch a Model Learn From Data
**Demystifying AI — Industry Event, May 2026**

Welcome — your first hands-on moment of the session.

## First: what is this thing?

If you've never opened a notebook before, two minutes of orientation:

**What you're looking at is a *Jupyter notebook*, opened in *Google Colab*.** A notebook is a document that mixes blocks of writing (like this one) with blocks of code. The code blocks have a little ▶ button — clicking it runs that code on Google's computer and shows the result right below. Think of it as a lab notebook for software: code, results, and notes interleaved on one page.

**Google Colab** is Google's free service for running notebooks in the browser. You don't install anything. You don't need to know Python. Google hands every attendee a temporary virtual computer to run the demo on. Close the tab when you're done; nothing to clean up.

**What you're about to do:** click `Runtime → Run all` once. Two animations will appear — one showing a line gradually settling onto its data as more data arrives, then another showing the same line getting yanked around by a handful of bad data points. **Each animation has a play button — click it when Mark says go.**

**You do not need to read or understand the code.** Every block is commented for the curious, but the demo is in the animations.

> **No GPU needed for this one.** Default Colab runtime (CPU) is fine — save your GPU allocation for the nanoGPT demo later in the session.

## What's in this notebook

- **Setup** — imports and a few knobs
- **Helpers** — three small functions: build fake data, fit a line, draw a plot
- **Phase 1 animation** — watch a line stabilize as data grows from 2 points to 1,000
- **Phase 2 animation** — inject 3 anomalies into a clean dataset; compare two fitting algorithms
- **Recap** — what this means and where the talk goes next


In [ ]:
# -------------------------------------------------------------------
# Setup — imports and the knobs that control the demo
# -------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, HuberRegressor


# === The "ground truth" line ===
# We KNOW the real relationship is y = 2.5x + 4. We're going to generate
# noisy data around that line, then see how well different fitting
# algorithms recover the true slope (m) and intercept (b).
# This is the simplest possible version of "training a model on data."
TRUE_M = 2.5      # true slope
TRUE_B = 4.0      # true y-intercept
NOISE_STD = 3.0   # how noisy each data point is around the true line

# Seed the random number generator so everyone in the room sees the
# SAME data on their screen (otherwise each laptop would draw different
# points and the narration wouldn't match).
RNG = np.random.default_rng(42)

print("Setup loaded. True line we're hunting: y = 2.5x + 4.")
print("Now run the next cell, knucklehead.")


In [ ]:
# -------------------------------------------------------------------
# Three small helper functions — make data, fit a line, draw a plot
# -------------------------------------------------------------------


def make_data(n, noise_std=NOISE_STD):
    """Generate n points scattered around the true line y = TRUE_M * x + TRUE_B."""
    x = np.linspace(0, 20, n)              # n evenly-spaced x values from 0 to 20
    noise = RNG.normal(0, noise_std, n)    # Gaussian noise added to each y
    y = TRUE_M * x + TRUE_B + noise
    return x, y


def fit_ols(x, y):
    """Fit a line using ORDINARY LEAST SQUARES — the textbook regression.

    OLS = "find the m and b that minimize the sum of SQUARED errors."
    The squaring is the key detail: one big error counts more than many
    small ones. That's why OLS is sensitive to outliers (Phase 2 shows this).
    """
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    return float(model.coef_[0]), float(model.intercept_)


def fit_huber(x, y):
    """Fit a line using HUBER REGRESSION — a "robust" alternative to OLS.

    Huber treats small errors like OLS does (squared), but treats LARGE
    errors LINEARLY instead of squaring them. That one change makes it
    much harder for a handful of bad points to drag the line off course.
    Used in any production setting where you expect dirty data.
    """
    model = HuberRegressor().fit(x.reshape(-1, 1), y)
    return float(model.coef_[0]), float(model.intercept_)


def draw(ax, x, y, m, b, title, extra_line=None, y_limits=None):
    """Draw one panel: scatter the points, plot the fit, plot the truth."""
    ax.clear()
    ax.scatter(x, y, s=30, alpha=0.7, label=f"data (n={len(x)})")

    # The line our model fit (red, solid)
    xs = np.array([0, 20])
    ax.plot(xs, m * xs + b, "r-", lw=2.2,
            label=f"OLS fit: y = {m:.2f}x + {b:.2f}")

    # Optional second line (used in Phase 2 to show Huber alongside OLS)
    if extra_line is not None:
        m2, b2, label2 = extra_line
        ax.plot(xs, m2 * xs + b2, "g--", lw=2.2, label=label2)

    # The true line we know we should be recovering (black dotted)
    ax.plot(xs, TRUE_M * xs + TRUE_B, "k:", lw=1, alpha=0.6,
            label=f"truth: y = {TRUE_M}x + {TRUE_B}")

    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 20)
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)


print("Helpers ready: make_data, fit_ols, fit_huber, draw.")
print("Phase 1 is next — that's the dramatic one. Go run it.")


## Phase 1 — Watch the line settle as data piles up

We're going to add data points one at a time and watch what happens to the fitted line. The animation cycles through twelve values of N:

**1, 2, 3, 4, 5, 10, 20, 50, 100, 1,000, 10,000, 1,000,000.**

**What to watch for:**

- **N = 1** — one data point. **No line exists yet.** You can't fit a model with one observation.
- **N = 2** — *now* a line exists (any two points determine one). With this particular random seed it *happens* to land close to the truth. **That's pure luck** — different random samples would give wildly different lines.
- **N = 3, 4** — line drifts a bit as the algorithm starts having to compromise.
- **N = 5** — **swings dramatically off course.** The intercept goes *negative*. This is what "small samples are unstable" actually looks like.
- **N = 10, 20** — line begins to settle.
- **N = 50 onward** — barely moves. *This is what "training converges" means.*
- **N = 1,000 → 1,000,000** — same line as N=100. **More data has diminishing returns once the signal is captured.**

> **Hit ▶ (play) below when Mark says go.** Pause on any frame to read the equation. The scrubber lets you jump directly between any two N values.


In [ ]:
# -------------------------------------------------------------------
# Phase 1 — Watch the line settle as data piles up
# Single-panel animation cycling through 12 values of N:
#   1, 2, 3, 4, 5, 10, 20, 50, 100, 1,000, 10,000, 1,000,000
#
# Each frame fits OLS on the first N points of a master dataset (so
# the data accumulates rather than being resampled — frame N=10
# contains the same points from frame N=2, plus 8 more).
# -------------------------------------------------------------------

from matplotlib import rc
from matplotlib.animation import FuncAnimation

# Render animations as inline HTML5 players (Colab's standard pattern)
rc("animation", html="jshtml")
rc("animation", embed_limit=50)  # MB cap on embedded animation size

# === Pre-generate ONE master dataset of 1,000,000 points ===
# Each frame uses the first N of these. That way the early points are
# stable across frames — the line settles as data piles up, instead
# of jumping wildly between unrelated random samples.
RNG = np.random.default_rng(42)
N_MAX = 1_000_000
x_master = RNG.uniform(0, 20, N_MAX)
y_master = TRUE_M * x_master + TRUE_B + RNG.normal(0, NOISE_STD, N_MAX)

# === The N values we'll cycle through ===
ns = [1, 2, 3, 4, 5, 10, 20, 50, 100, 1_000, 10_000, 1_000_000]

# === Pre-compute fits + scatter subsamples for each frame ===
# Fitting OLS on 1M points is fast (closed-form), but scattering 1M
# points is slow. We use a 2,000-point random subsample for the
# scatter plot whenever N is huge — the *fit* still uses every point.
frame_data = []
plot_rng = np.random.default_rng(0)  # separate RNG so subsample is stable
for n in ns:
    x = x_master[:n]
    y = y_master[:n]
    if n >= 2:
        m, b = fit_ols(x, y)
    else:
        m, b = None, None  # can't fit a line through one point
    if n > 2000:
        idx = plot_rng.choice(n, 2000, replace=False)
        x_plot, y_plot = x[idx], y[idx]
    else:
        x_plot, y_plot = x, y
    frame_data.append((n, m, b, x_plot, y_plot))

# === Build the animation ===
fig, ax = plt.subplots(figsize=(11, 7))


def update_phase1(frame):
    n, m, b, x_plot, y_plot = frame_data[frame]
    ax.clear()

    # Scatter — smaller and more transparent at huge N to avoid a solid blob
    if n > 2000:
        ax.scatter(x_plot, y_plot, s=6, alpha=0.25,
                   label=f"data (n = {n:,}; showing 2,000)")
    else:
        ax.scatter(x_plot, y_plot, s=55, alpha=0.75,
                   label=f"data (n = {n:,})")

    xs = np.array([0, 20])

    # Truth line (always visible — the answer we're trying to find)
    ax.plot(xs, TRUE_M * xs + TRUE_B, "k:", lw=1.3, alpha=0.6,
            label=f"truth: y = {TRUE_M}x + {TRUE_B}")

    # Fitted line — only exists for N >= 2
    if m is not None:
        ax.plot(xs, m * xs + b, "r-", lw=2.8, label="OLS fit")
        title = f"N = {n:,}      →      y = {m:.3f}x + {b:.3f}"
    else:
        title = "N = 1      →      can't draw a line through one point"

    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 20)
    ax.set_ylim(-10, 70)
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(alpha=0.3)


# 1800 ms per frame — slow enough to read the equation in the title.
anim_phase1 = FuncAnimation(
    fig, update_phase1, frames=len(frame_data), interval=1800, repeat=False
)

plt.close(fig)
anim_phase1


**What you just watched:**

- **One observation = no model.** You can describe a single data point; you can't generalize from it.
- **Two observations = a model that's probably wrong.** A line exists by definition through any two points, but it tells you nothing about the population those points were drawn from.
- **Three observations = the algorithm has to compromise.** First moment of real *fitting* — the line can no longer pass through every point. It has to find the best balance.
- **By N=50** the line has essentially landed on the truth. By N=100 it has *barely moved*.
- **N = 1,000,000.** Same line as N=100. Different decimal places, same answer.

> *"More data helps, then stops helping. Once you've captured the signal, additional samples are confirmation, not new information. The hard question stops being 'do we have enough data' and starts being 'do we have the **right** data.' That distinction is where most enterprise AI projects live or die."*


## Phase 2 — Bad data tilts the line

Real data is messy. Sensor glitches, typos, fat-fingered entries. What happens when a clean 100-point dataset picks up a few wildly-wrong points?

Below, we start with the clean dataset and progressively inject 1, 2, then 3 anomalies. We fit the same data two different ways:

- **OLS** (red, solid) — what most regressions use by default. Sensitive to outliers.
- **Huber** (green, dashed) — a "robust" alternative that down-weights outliers instead of squaring them.

**Hit ▶ on the animation below.** Watch the red line tilt. Watch the green line stay put.


In [ ]:
# -------------------------------------------------------------------
# Phase 2 — Animated anomaly injection, OLS vs Huber
# Each frame adds one more anomaly to a clean 100-point dataset.
# Output is an inline HTML5 player with play/pause/scrub controls.
# -------------------------------------------------------------------

# Start with a clean 100-point dataset
x_clean, y_clean = make_data(100)

# Three anomalies — chosen to be obviously, comically wrong
# (a real-world sensor glitch or data-entry typo would look similar)
anomalies_x = np.array([3.0, 8.0, 14.0])
anomalies_y = np.array([95.0, -35.0, 120.0])

fig, ax = plt.subplots(figsize=(11, 7))


def update_phase2(k):
    # k = number of anomalies injected so far (0, 1, 2, 3)
    xk = np.concatenate([x_clean, anomalies_x[:k]])
    yk = np.concatenate([y_clean, anomalies_y[:k]])

    # Fit OLS to the (clean + k anomalies) data
    m_ols, b_ols = fit_ols(xk, yk)

    # Also fit Huber once there's at least one anomaly to ignore
    extra = None
    if k >= 1:
        m_h, b_h = fit_huber(xk, yk)
        extra = (m_h, b_h, f"Huber (robust): y = {m_h:.2f}x + {b_h:.2f}")

    label = "anomaly" if k == 1 else "anomalies"
    draw(ax, xk, yk, m_ols, b_ols,
         f"Phase 2 — {k} {label} added (N = {len(xk)})",
         extra_line=extra, y_limits=(-50, 140))


# Slower interval here (2500 ms) so the room has time to register
# the tilt before the next anomaly lands.
anim_phase2 = FuncAnimation(
    fig, update_phase2, frames=len(anomalies_x) + 1, interval=2500, repeat=False
)

plt.close(fig)
anim_phase2


**What you should be seeing above:**

- **0 anomalies:** OLS sits right on top of the truth. No drama.
- **1 anomaly:** ONE bad point pulled the red OLS line off course. Huber barely moved.
- **2 anomalies:** OLS now visibly tilted. Huber still on the truth.
- **3 anomalies:** OLS is meaningfully wrong. Huber is still right.

> *"The algorithm determines how your model fails when reality is messy. When a vendor shows you a benchmark, they've shown you the red line on clean data. The real question — the one that predicts whether their model survives your production environment — is what happens when you hand it your messy data."*

---

## Where this goes next

Everything you just watched was **two numbers** — m and b — being adjusted to fit data. GPT-4 has **1.76 trillion** of those numbers. The adjustment procedure is the same. The only things that change are the function's shape, the parameter count, and how much compute you throw at it.

Keep that image in your head for the rest of the talk — because every architecture we're about to discuss (tokenization, embeddings, attention, transformers, the nanoGPT demo you'll run later) is a more elaborate version of this exact picture.
